# 01 — EDA: Synthetic Resume Dataset

**Project H17 — Fair AI Hiring with Bias Audit.** 5,000 candidates with hire-label, demographic and skill features. A mild proxy bias is injected via `prior_employer_tier` — exactly the failure mode a fairness audit should catch.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
df = pd.read_parquet('../data/processed/resumes.parquet')
df.head()

## 1. Shape and target prevalence

In [ ]:
print(f'rows: {len(df):,}   cols: {df.shape[1]}')
print(f'positive rate (hire_label): {df["hire_label"].mean():.3f}')
print('\nbalance per gender:')
print(df.groupby('gender')['hire_label'].agg(['count', 'mean']).round(3))
print('\nbalance per nationality_group:')
print(df.groupby('nationality_group')['hire_label'].agg(['count', 'mean']).round(3))

## 2. Hire rate per protected attribute

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ['gender', 'nationality_group']):
    sub = df.groupby(col)['hire_label'].mean().sort_values()
    sns.barplot(x=sub.values, y=sub.index, palette='Set2', ax=ax)
    ax.set_title(f'Hire rate by {col}')
    ax.set_xlim(0, 0.7)
plt.tight_layout(); plt.show()

## 3. Detect the proxy bias channel — `prior_employer_tier` × gender

In [ ]:
tab = pd.crosstab(df['gender'], df['prior_employer_tier'], normalize='index').round(3)
print('share of each gender at each tier:'); print(tab)
fig, ax = plt.subplots(figsize=(8, 4))
tab.plot(kind='bar', stacked=True, ax=ax, colormap='YlOrBr')
ax.set_title('Prior-employer tier distribution by gender — proxy bias channel')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 4. Years of experience and education distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['years_experience'], bins=25, color='#1f77b4', ax=axes[0])
axes[0].set_title('years_experience')
order = ['High School', 'Bachelor', 'Master', 'PhD']
sns.countplot(data=df, x='education_level', order=order, palette='Set3', ax=axes[1])
axes[1].set_title('education_level')
plt.tight_layout(); plt.show()

## 5. Hire rate by tier × gender (the unfairness, made visible)

In [ ]:
tier_g = df.groupby(['prior_employer_tier', 'gender'])['hire_label'].mean().reset_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=tier_g, x='prior_employer_tier', y='hire_label', hue='gender',
            palette=['#1f77b4', '#fb6a4a'], ax=ax)
ax.set_title('Hire rate by tier and gender'); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 6. Skill-feature density — sparse positive (TF-IDF-like)

In [ ]:
from fair_hiring.features import skill_columns
sys.path.insert(0, '../src')
from fair_hiring.features import skill_columns
skill_cols = skill_columns(df)
print(f'skill columns: {len(skill_cols)}')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[skill_cols].sum(axis=1), bins=30, color='#2ca02c', ax=axes[0])
axes[0].set_title('Total skill activation per candidate')
sns.histplot(df[skill_cols].values.ravel(), bins=40, color='#9467bd', ax=axes[1])
axes[1].set_title('Per-cell skill value distribution')
plt.tight_layout(); plt.show()

## 7. Correlation heatmap — top skill features

In [ ]:
C = df[skill_cols[:12]].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(C, annot=False, cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation among first 12 skill features')
plt.tight_layout(); plt.show()

## 8. Group-conditional hire rates — gender × education

In [ ]:
ge = df.groupby(['education_level', 'gender'])['hire_label'].mean().reset_index()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=ge, x='education_level', y='hire_label', hue='gender',
            order=order, palette=['#1f77b4', '#fb6a4a'], ax=ax)
ax.set_title('Hire rate by education × gender')
plt.tight_layout(); plt.show()

## 9. Implications for modelling and audit
- Marginal hire rate gap between genders is small — but the `prior_employer_tier` distribution is shifted.
- A learner that drops `gender` will still pick up the unfairness via `prior_employer_tier` (the proxy). The whole point of the audit downstream.
- We will measure selection rate, TPR, and FPR per gender and per nationality_group; equalised-odds postprocessing will flatten the TPR/FPR gap.